## ***Import Libraries***

All required libraries are imported here.

In [1]:
import pyodbc
import pandas as pd
from sqlalchemy import create_engine
import urllib
import sys
import os
sys.path.append(r'C:\Users\ujay\Desktop\CodeAlpha_DataAnalysis')

from config import SERVER, DATABASE

## ***Load Data from SQL Server***

Data is loaded directly from the `cleaned` schema in the
`ecommerce_analysis` SQL Server database using `pd.read_sql()`.


In [2]:
# connect to database
conn = pyodbc.connect(
    f'DRIVER={{ODBC Driver 17 for SQL Server}};'
    f'SERVER={SERVER};'
    f'DATABASE={DATABASE};'
    f'Trusted_Connection=yes;'
)

print("Connection successful!")

Connection successful!


In [3]:
# load tabel from the schema
orders = pd.read_sql("SELECT * FROM cleaned.orders", conn)
customers = pd.read_sql("SELECT * FROM cleaned.customers", conn)
order_items = pd.read_sql("SELECT * FROM cleaned.order_items", conn)
payments = pd.read_sql("SELECT * FROM cleaned.payments", conn)
reviews = pd.read_sql("SELECT * FROM cleaned.reviews", conn)
products = pd.read_sql("SELECT * FROM cleaned.products", conn)
sellers = pd.read_sql("SELECT * FROM cleaned.sellers", conn)
geolocation = pd.read_sql("SELECT * FROM cleaned.geolocation", conn)
product_category = pd.read_sql("SELECT * FROM cleaned.category_translation", conn)

C:\Users\ujay\AppData\Local\Temp\ipykernel_9520\656148436.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  orders = pd.read_sql("SELECT * FROM cleaned.orders", conn)
C:\Users\ujay\AppData\Local\Temp\ipykernel_9520\656148436.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  customers = pd.read_sql("SELECT * FROM cleaned.customers", conn)
C:\Users\ujay\AppData\Local\Temp\ipykernel_9520\656148436.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  order_items = pd.read_sql("SELECT * FROM cleaned.order_items", conn)
C:\Use

In [4]:
# Display all column names for each table
tables = {
    'orders': orders,
    'customers': customers,
    'order_items': order_items,
    'payments': payments,
    'reviews': reviews,
    'products': products,
    'sellers': sellers,
    'geolocation': geolocation,
    'product_category': product_category
}

for name, df in tables.items():
    print(f"\n{'.'*50}")
    print(f"{name} :  {df.shape[0]} rows x {df.shape[1]} columns")
    print(f"Columns: {list(df.columns)}")



..................................................
orders :  99441 rows x 8 columns
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

..................................................
customers :  99441 rows x 5 columns
Columns: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

..................................................
order_items :  112650 rows x 7 columns
Columns: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

..................................................
payments :  103886 rows x 5 columns
Columns: ['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

..................................................
reviews :  99224 rows x 7 columns
Columns: ['review_id', 'order_id', '

# ***Data Cleaning***


### ***Fix Column Names***

Several dataframes contained column names wrapped in double quotes (e.g. `"order_id"` instead of `order_id`).
This was caused during the SQL Server import process.


### **Data Value cleaning**

Row values were wrapped in quotes (e.g., `'"value"'`), blocking numeric conversion and joins.

**Solution:** Applied a `.str.strip('"')` function to targeted columns.


In [5]:
#  list af all dataframe
dataframes = [orders, customers, order_items, payments, 
              reviews, products, sellers, geolocation, product_category]

def clean_data(dfs):
    for df in dfs:
        # Fix Column Names
        df.columns = df.columns.str.replace('"', '', regex=False)
        
        # Fix Row Data 
        cols = df.select_dtypes(include=['object']).columns
        for col in cols:
            df[col] = df[col].str.replace('"', '', regex=False)
clean_data(dataframes)

In [6]:
# orders.head()
# orders.head()
# customers.head()
# order_items.head()
# payments.head()
# reviews.head().
# products.head()
# sellers.head()
# geolocation.head()
product_category.head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


## Wrangling All Dataframes

Individual wrangle functions were defined and applied to each of the 9 dataframes to:
- **Drop** unnecessary columns (timestamps, redundant IDs, physical dimensions)
- **Cast** identifier and categorical columns to `string` dtype
- **Convert** date columns to `datetime64` and numeric columns to `float` or `int`

In [7]:

# Orders: Dropped internal timestamps and update dtypes
def wrangle_orders(df):
    cols_to_drop = ['order_approved_at','order_delivered_carrier_date']
    df = df.drop(columns=cols_to_drop, errors='ignore')
    df[['order_id', 'customer_id', 'order_status']] = df[['order_id', 'customer_id', 'order_status']].astype(str)
    date_cols = ['order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date']
    df[date_cols] = df[date_cols].apply(pd.to_datetime, errors='coerce')
    return df

# Customers: Dropped redundant unique_id and update datatypes
def wrangle_customers(df):
    cols_to_drop = ['customer_unique_id']
    df = df.drop(columns=cols_to_drop, errors='ignore')
    
    df[['customer_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']] = \
        df[['customer_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']].astype(str)
    return df

# Order Items: Dropped limit date and update dtypes
def wrangle_order_items(df):
    cols_to_drop = ['shipping_limit_date']
    df = df.drop(columns=cols_to_drop, errors='ignore')
    df[['order_id', 'order_item_id', 'product_id', 'seller_id']] = \
        df[['order_id', 'order_item_id', 'product_id', 'seller_id']].astype(str)
    df[['price', 'freight_value']] = df[['price', 'freight_value']].apply(pd.to_numeric, errors='coerce')
    return df

# Payments: drop a column and fix datatyoes
def wrangle_payments(df):
    cols_to_drop = ['payment_sequential']
    df = df.drop(columns=cols_to_drop, errors='ignore')
    df[['order_id', 'payment_type']] = df[['order_id', 'payment_type']].astype(str)
    df['payment_installments'] = pd.to_numeric(df['payment_installments'], errors='coerce').fillna(0).astype(int)
    df['payment_value'] = pd.to_numeric(df['payment_value'], errors='coerce')
    return df

# Reviews
def wrangle_reviews(df):
    cols_to_drop = ['review_comment_title', 'review_creation_date', 'review_answer_timestamp']
    df = df.drop(columns=cols_to_drop, errors='ignore')
    df[['review_id', 'order_id', 'review_comment_message']] = \
        df[['review_id', 'order_id', 'review_comment_message']].astype(str)

    df['review_score'] = pd.to_numeric(df['review_score'], errors='coerce').fillna(0).astype(float)
    return df

# Products
def wrangle_products(df):
    cols_to_drop = ['product_name_lenght', 'product_description_lenght', 'product_weight_g', 
                    'product_length_cm', 'product_height_cm', 'product_width_cm']
    df = df.drop(columns=cols_to_drop, errors='ignore')
    df[['product_id', 'product_category_name']] = df[['product_id', 'product_category_name']].astype(str)
    df['product_photos_qty'] = pd.to_numeric(df['product_photos_qty'], errors='coerce')
    return df

# Sellers
def wrangle_sellers(df):
    df[['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']] = \
        df[['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']].astype(str)
    return df

#Geolocation
def wrangle_geolocation(df):
    df[['geolocation_zip_code_prefix', 'geolocation_city', 'geolocation_state']] = \
        df[['geolocation_zip_code_prefix', 'geolocation_city', 'geolocation_state']].astype(str)
    df[['geolocation_lat', 'geolocation_lng']] = \
        df[['geolocation_lat', 'geolocation_lng']].apply(pd.to_numeric, errors='coerce')
    return df

# Product Category
def wrangle_product_category(df):
    df[['product_category_name', 'product_category_name_english']] = \
        df[['product_category_name', 'product_category_name_english']].astype(str)
    return df

orders = wrangle_orders(orders)
customers = wrangle_customers(customers)
order_items = wrangle_order_items(order_items)
payments = wrangle_payments(payments)
reviews = wrangle_reviews(reviews)
products = wrangle_products(products)
sellers = wrangle_sellers(sellers)
geolocation = wrangle_geolocation(geolocation)
product_category = wrangle_product_category(product_category)


In [8]:
reviews.tail(20)

,review_id,order_id,review_score,review_comment_message
99204,8c4f17f8e6eb53b9af28b989e2c92a59,d9647302d75348f06e63f8444ef95544,5.0,produto de otima qualidade
99205,b7ce5a0a606b61681caee5b8cdd3fcbd,0690b2d9b983cf580f407531575622c5,1.0,"não chegou ainda a encomenda, o motivo deve se..."
99206,a0d6f68abffcd4c639baf591b51110be,55d18e6e778d8bbc28aff36ab14104ba,5.0,None
99207,8a2dcd97acec47f99ab1f58d138e6c35,2bb9ca1905f76e4e33fc5ae85351dbee,5.0,None
99208,db7011f989e4281b6e76bd3d419075f7,e94197623546186601aa880a31269468,4.0,None
99209,9dc3f89c8f5172b157cc41cb4b2b4f6f,d115642fba3060e4082472bb2801b111,1.0,ainda não recebi o produto???????
99210,0bb958acd9e40d9348635c75013f7b19,4f9f24f1dd52ecdf0684a63eaf36c51b,4.0,None
99211,b96778286497f47942bb9d1f588545b8,6895b2b445ba6fc95b205ec5d0d5f25f,5.0,None
99212,6bd7685fa37d93f87a915a4b032d4b6d,e13713108caa1e5a2604672467aa9d8a,1.0,"Não tenho certeza, pois ainda não recebi o pro..."
99213,4a72f90fc8932e9c6330115587c2eace,a0fe3a166733b6b4c91bca9b1cd91972,5.0,None


In [9]:
products.head()

,product_id,product_category_name,product_photos_qty
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,1.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,1.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,1.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,1.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,4.0


## ***Missing Values in orders***



In [10]:
# check for missing missing values
dataframes = [orders, customers, order_items, payments, 
              reviews, products, sellers, geolocation, product_category]

names = ['orders', 'customers', 'order_items', 'payments', 
         'reviews', 'products', 'sellers', 'geolocation', 'product_category']
# Check missing values across all dataframes
for name, df in zip(names, dataframes):
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing) > 0:
        print(f"\n{name.upper()} missing values:")
        print(missing)
    else:
        print(f"\n{name.upper()}: No missing values")


ORDERS missing values:
order_delivered_customer_date    2965
dtype: int64

CUSTOMERS: No missing values

ORDER_ITEMS: No missing values

PAYMENTS: No missing values

REVIEWS: No missing values

PRODUCTS missing values:
product_photos_qty    610
dtype: int64

SELLERS: No missing values

GEOLOCATION: No missing values

PRODUCT_CATEGORY: No missing values


### fill missing vlaues in orders

In [11]:
# Fill missing product_photos_qty with 0
products['product_photos_qty'] = products['product_photos_qty'].fillna(0)

# Fill missing order_delivered_customer_date with 'Not Delivered'
orders['order_delivered_customer_date'] = orders['order_delivered_customer_date'].fillna('Not Delivered')

# Verify no more missing values
print("Orders missing:", orders.isnull().sum().sum())
print("Products missing:", products.isnull().sum().sum())

Orders missing: 0
Products missing: 0


### create a flag column is_delivered in  the orders table

In [12]:
orders['is_delivered'] = orders['order_delivered_customer_date'].apply(
    lambda x: False if x == 'Not Delivered' else True)

In [13]:
check_delivery_status = orders.groupby(['order_status', 'is_delivered']).size().reset_index(name = 'count')
check_delivery_status

,order_status,is_delivered,count
0,approved,False,2
1,canceled,False,619
2,canceled,True,6
3,created,False,5
4,delivered,False,8
5,delivered,True,96470
6,invoiced,False,314
7,processing,False,301
8,shipped,False,1107
9,unavailable,False,609


## Data Quality Issue - Delivered Orders with Missing Delivery Date

### Observation
After creating the `is_delivered` flag column, a cross-check between `order_status` 
and `is_delivered` revealed an inconsistency:

| order_status | is_delivered | count |
|---|---|---|
| delivered | False | 8 |
| canceled | True | 6 |

### Issues Found
1. **8 orders** have `order_status = delivered` but `is_delivered = False` - meaning 
   they were marked as delivered in the system but have **no delivery date recorded**.
2. **6 orders** have `order_status = canceled` but `is_delivered = True` - meaning 
   they were cancelled but still have a **delivery date recorded**. This is contradictory.


In [14]:
# delivered status but is_delivered = False — correct to True
orders.loc[orders['order_status'] == 'delivered', 'is_delivered'] = True

# canceled status but is_delivered = True — correct to False
orders.loc[orders['order_status'] == 'canceled', 'is_delivered'] = False

# Verify fix
check = orders.groupby(['order_status', 'is_delivered']).size().reset_index(name='count')
print(check)

  order_status  is_delivered  count
0     approved         False      2
1     canceled         False    625
2      created         False      5
3    delivered          True  96478
4     invoiced         False    314
5   processing         False    301
6      shipped         False   1107
7  unavailable         False    609


## State Code Mapping

Brazilian state abbreviations (e.g. `SP`, `RJ`) were mapped to their full names 
across the `customers`, `sellers`, and `geolocation` tables to improve readability 
and support accurate geographic visualization in Power BI.````m

In [15]:
# Brazilian state codes to full names
state_mapping = {
    'AC': 'Acre',
    'AL': 'Alagoas',
    'AP': 'Amapá',
    'AM': 'Amazonas',
    'BA': 'Bahia',
    'CE': 'Ceará',
    'DF': 'Distrito Federal',
    'ES': 'Espírito Santo',
    'GO': 'Goiás',
    'MA': 'Maranhão',
    'MT': 'Mato Grosso',
    'MS': 'Mato Grosso do Sul',
    'MG': 'Minas Gerais',
    'PA': 'Pará',
    'PB': 'Paraíba',
    'PR': 'Paraná',
    'PE': 'Pernambuco',
    'PI': 'Piauí',
    'RJ': 'Rio de Janeiro',
    'RN': 'Rio Grande do Norte',
    'RS': 'Rio Grande do Sul',
    'RO': 'Rondônia',
    'RR': 'Roraima',
    'SC': 'Santa Catarina',
    'SP': 'São Paulo',
    'SE': 'Sergipe',
    'TO': 'Tocantins'
}

# Apply to all dataframes that have a state column
customers['customer_state'] = customers['customer_state'].map(state_mapping)
sellers['seller_state'] = sellers['seller_state'].map(state_mapping)
geolocation['geolocation_state'] = geolocation['geolocation_state'].map(state_mapping)


## Sentiment Label Creation (Review table)

Review scores (1–5) were mapped to sentiment labels - `Negative` (1–2), `Neutral` (3), 
and `Positive` (4–5) - to prepare the reviews table for Task 4 sentiment analysis.

In [16]:
# Map review scores to sentiment labels
reviews['sentiment'] = reviews['review_score'].map({
    1: 'Negative',
    2: 'Negative',
    3: 'Neutral',
    4: 'Positive',
    5: 'Positive'
})

# Verify
print(reviews['sentiment'].value_counts())

sentiment
Positive    76470
Negative    14575
Neutral      8179
Name: count, dtype: int64


### check the payment types


**Remark**: All good

In [17]:
# payments['payment_type'].unique()
payments['payment_type'].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

## Product Category Alignment

Compared product category names across the `products` and `product_category` tables to identify mismatches.

- `products` table contained **73 unique categories** (excluding `unknown`)
- `product_category` translation table contained **71 categories**
- **2 categories** were found in `products` but missing from the translation table: `pc_gamer` and `portateis_cozinha_e_preparadores_de_alimentos`
- **1 empty string** was found and replaced with `unknown` to handle products with no category assigned
- The 2 missing categories were manually added to the translation table with their English equivalents

In [18]:
# Sort and compare product category names in both tables
products_cats = sorted(products['product_category_name'].unique())
category_cats = sorted(product_category['product_category_name'].unique())

# len(products_cats) == len(category_cats)
len(products_cats)
len(category_cats)

# # Find mismatches using set difference
in_products_not_in_category = set(products_cats) - set(category_cats)
in_category_not_in_products = set(category_cats) - set(products_cats)

# Replace empty string with 'unknown'
products['product_category_name'] = products['product_category_name'].replace('', 'unknown')

# Add missing categories to product_category table
missing_cats = pd.DataFrame({
    'product_category_name': ['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'],
    'product_category_name_english': ['pc gamer', 'portable kitchen and food preparers']
})
product_category = pd.concat([product_category, missing_cats], ignore_index=True)

# Verify mismatches are gone
products_cats = sorted(products['product_category_name'].unique())
category_cats = sorted(product_category['product_category_name'].unique())

in_products_not_in_category = set(products_cats) - set(category_cats)
in_category_not_in_products = set(category_cats) - set(products_cats)

print("In products not in category:", in_products_not_in_category)
print("In category not in products:", in_category_not_in_products)

In products not in category: {'unknown'}
In category not in products: set()


In [21]:
# in_category_not_in_products
# in_products_not_in_category
product_category.tail()
# products.tail()

,product_category_name,product_category_name_english
68,fraldas_higiene,diapers_and_hygiene
69,fashion_roupa_infanto_juvenil,fashion_childrens_clothes
70,seguros_e_servicos,security_and_services
71,pc_gamer,pc gamer
72,portateis_cozinha_e_preparadores_de_alimentos,portable kitchen and food preparers


## Duplicate Row Check

A reusable function `check_and_drop_duplicates()` was built to identify and remove duplicate 
rows across the five tables where uniqueness is expected: `orders`, `customers`, `reviews`, 
`sellers`, and `products`. 

No duplicates were found in any table  all five passed the check cleanly and required no action.

In [26]:
def check_and_drop_duplicates(tables_dict):
    """
    Checks for duplicate rows in each dataframe,
    reports findings, and drops duplicates where found.
    
    Args:
        tables_dict: dictionary of {table_name: dataframe}
    
    Returns:
        dictionary of cleaned dataframes
    """
    cleaned = {}
    
    for name, df in tables_dict.items():
        dupes = df.duplicated().sum()
        if dupes > 0:
            print(f"{name}: {dupes} duplicates found - dropping...")
            cleaned[name] = df.drop_duplicates().reset_index(drop=True)
            print(f"{name}: cleaned. Now {len(cleaned[name])} rows")
        else:
            print(f"{name}: no duplicates found - skipping")
            cleaned[name] = df
    
    return cleaned

# Run the function
tables_to_check = {
    'orders': orders,
    'customers': customers,
    'reviews': reviews,
    'sellers': sellers,
    'products': products
}

cleaned_tables = check_and_drop_duplicates(tables_to_check)

# Reassign back to original variables
orders = cleaned_tables['orders']
customers = cleaned_tables['customers']
reviews = cleaned_tables['reviews']
sellers = cleaned_tables['sellers']
products = cleaned_tables['products']

print("\nDone.")

orders: no duplicates found — skipping
customers: no duplicates found — skipping
reviews: no duplicates found — skipping
sellers: no duplicates found — skipping
products: no duplicates found — skipping

Done.


## Save Cleaned Data to SQL Server

All 9 cleaned dataframes were saved back to SQL Server under the `cleaned` schema using 
SQLAlchemy. A reusable `upload_cleaned_data()` function was built to handle the upload, 
using `if_exists='replace'` to overwrite any previously cleaned tables and `chunksize=1000` 
to handle large datasets like `geolocation` (1M+ rows) efficiently.

This completes the data cleaning pipeline. All subsequent analysis notebooks will read 
from the `cleaned` schema rather than the `raw` schema.

In [28]:
# Format the connection string
params = urllib.parse.quote_plus(
    f'DRIVER={{ODBC Driver 17 for SQL Server}};'
    f'SERVER={SERVER};'
    f'DATABASE={DATABASE};'
    f'Trusted_Connection=yes;'
)

# Create the SQLAlchemy engine
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

# Mapping dataframes to their SQL table names
df_map = {
    "orders": orders,
    "customers": customers,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "sellers": sellers,
    "geolocation": geolocation,
    "product_category": product_category
}

# upload function with schema support
def upload_cleaned_data(mapping, sql_engine):
    for table_name, df in mapping.items():
        print(f"Uploading {table_name} to 'cleaned' schema...")
        # Added schema='cleaned' to target the cleaned schema
        df.to_sql(
            name=table_name, 
            con=sql_engine, 
            schema='cleaned', 
            if_exists='replace', 
            index=False,
            chunksize=1000 # Helpful for industrial-sized datasets
        )
    print("Database update to 'cleaned' schema complete!")

# Run the upload
upload_cleaned_data(df_map, engine)

Uploading orders to 'cleaned' schema...


C:\ProgramData\anaconda3\Lib\site-packages\pandas\io\sql.py:1648: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


Uploading customers to 'cleaned' schema...
Uploading order_items to 'cleaned' schema...
Uploading payments to 'cleaned' schema...
Uploading reviews to 'cleaned' schema...
Uploading products to 'cleaned' schema...
Uploading sellers to 'cleaned' schema...
Uploading geolocation to 'cleaned' schema...
Uploading product_category to 'cleaned' schema...
Database update to 'cleaned' schema complete!
